#### Middleware

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")

In [5]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage , SystemMessage

agent = create_agent(
    model="google_genai:gemini-3.5-flash",
    checkpointer=InMemorySaver() ,
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-3.5-flash",
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [6]:
config = {"configurable":{"thread_id":"test-1"}}

questions = [
    "What is 2+2?",
    "What is 5+3?",
    "What is 10-4?",
    "What is 6*2?",
    "What is 15/3?",
    "What is 9+7?"
]


In [12]:
for q in questions:
    response = agent.invoke({
        "messages":[HumanMessage(
            content=q
        )] 
    },config)

    print(f"messages:{response}")
    print(f"messages:{len(response["messages"])}")

messages:{'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='d5a52142-ea6d-40dd-90cd-f2d9e186075c'), AIMessage(content=[{'type': 'text', 'text': '2 + 2 is 4.', 'extras': {'signature': 'EpECCo4CARFNMg9krdb7/sLu/Jm/QkrbeTuShJIEGu5mQqWA+XGvxpY9RUkGho/VBWDPVJfn6yddmu2ogNlLdYKG60cBSSiqre7uSGyjXJntG7jpOYNIY8Lp7yG4tHAquiUq+GtI/50QGMSGRi8+BfURbenANQeDNy5Cpi71tAsCm3n1Iw3jcoamnj4ltyJYraq0OgefEKfAqsoVL1xF7q65P/DJSIWZuXIFTJFYwEpX6VNL/ViuIY3nsVUeKwm8uazd/qPvZOvK5B63qLqRTCTtmF7wATNlQjKrjHADJ2HUaN7QhgTI5x5GSy9mR+N6eqfGrEIBlUEeN4d/qqePSor4Uok201xPfj2D7fXu/DX/1/al'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f5309-a3f5-7e73-860b-6be23635522b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 79, 'total_tokens': 87, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'re

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotel(city:str)->str:
    """search hotels-returns lang response to use more tokens"""
    return f"""Hotels in {city}: 
            1. grand hotel -5 star , $350/night , spa , pool , gym 
            2. city inn -4 star , $180/night , bussiness center 
            3. budget stay -3 star , $75/night , free wifi  
    """

agent = create_agent(
     model="google_genai:gemini-2.5-flash",
     tools=[search_hotel] ,\
     checkpointer=InMemorySaver(),
     middleware=[SummarizationMiddleware(
        model="google_genai:gemini-2.5-flash",
        trigger=("tokens",550),
        keep=("tokens",200)
     )]

)

In [ ]:
config = {"configurable":{"thread_id":"test-1"}}


def count_token(messages):
    total_chars = 0

    for m in messages:
        total_chars += len(str(m.content))

    return total_chars

In [ ]:
cities = ["paris", "London", "tokyo", "new york", "dubai", "singapore"]

for city in cities:
    response = agent.invoke(
        {
            "messages": [
                HumanMessage(
                    content=f"find hotels in {city}"
                )
            ]
        },
        config=config
    )

    tokens = count_token(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(response["messages"])

#### human in the loop middleware

In [10]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool

@tool
def read_email(email_id:str)->str:
    """mock funcion to read an email by its ID"""
    return f"emall content for ID :{email_id}"

@tool
def send_email(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email"""
    return f"Email sent to {recipient} with subject '{subject}'"


In [11]:
agent = create_agent(
    model="google_genai:gemini-2.5-flash" ,
    tools=[read_email,send_email] ,
    checkpointer=InMemorySaver() ,
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email":{
                    "allowed_decisions":["approve","edit","reject"]
                } ,
                "read_email":False
            }
        )
    ]
)

In [15]:
from langchain_core.messages import HumanMessage
config={"configurable":{"thread_id":"test_approve"}}

result = agent.invoke(
    {"messages":[
        HumanMessage(content="sent email to jhon@email.com with subject 'hello' and body 'how are you'")
    ]} ,
    config=config
)

print(result)

{'messages': [HumanMessage(content="sent email to jhon@email.com with subject 'hello' and body 'how are you'", additional_kwargs={}, response_metadata={}, id='5e59bd82-c888-4d48-a11a-9c51a63c7b98'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'send_email', 'arguments': '{"body": "how are you", "recipient": "jhon@email.com", "subject": "hello"}'}, '__gemini_function_call_thought_signatures__': {'b0e8c33b-8da4-4dee-8b8a-474e46c5612c': 'CvIEARFNMg9SQAlUvgmBXcFEhljr0pscHyVDvraX4scH9ukZeyoZ6nM9ofov80FuNKXPerxP884psvPAA8Opo/2rAs4ur+YrCMeTtdyCtkddmDIL+mgwWRxtQbd1povC91b5XmfU3duV6JWbgJQ7sGpKLRvOvd9US39J1Mn3DNB+IGxV5eIvgmvVo0jPuNvt9XoPhGGIp28klU25+41tDxx5nNLPX8vEifBEh+/1lOejti2FdN8VBxbiA4QJrvo3shw5As1ou6VKSrtMZeMwcvxKQGqrQ6qpk0Uo5T2bc/qRM09cg1QxCxXXvQi09KlxAFsmm+opyMU5JX7YSG69UOqhnxtX/DLaM5IWJjwd0OQHRP/otfq8oinjU3e8QjrZY1H8KwUz02zw6FAYYg4dN6vGV6OoZ7XxoMHGp4TQllUsw5gZry8EpJpQI0enq0YQcIuTsxyoJZJ4hUlqq2vy56uIKFJlWUsla4BobqgqMak75jHZclnbwVZxPxUvShrywbH1iw84zFbL8JUvUYzFl7v4Xl3

In [16]:
from langgraph.types import Command

if"__interrupt__" in result:
    print("paused:approving")
    
    result=agent.invoke(
        Command(
            resume={
            "decisions": [
                {"type": "approve"}
            ]
}
        ),config=config
    )
    print(f"result:{result["messages"][-1].content}")

paused:approving
result:I have sent an email to jhon@email.com with the subject 'hello' and body 'how are you'.
